# IMDb Sentiment Classification with BERT — **Student Version**

This notebook has been adapted for classroom use. Wherever original code has been removed, you will find **TODO** cells with a clear explanation of the required task.

**Rules of the game:**
- Fill each TODO cell following the instructions written in English.
- Keep variable names as indicated (e.g., `raw_datasets`, `tokenizer`, `tokenized_datasets`, `data_collator`, `model`, `training_args`, `trainer`) so later cells work smoothly.
- You may consult the official 🤗 Transformers and Datasets documentation.
- Feel free to add new cells for experiments, but do not delete the TODO prompts.

> Tip: if you run out of GPU memory, reduce `per_device_train_batch_size` or number of epochs.

# BERT Fine-Tuning on Kaggle IMDB (via `kagglehub`)

This Colab-ready notebook downloads the IMDB 50K movie reviews dataset **without** needing `kaggle.json` (public dataset) using **kagglehub**, fine‑tunes **`bert-base-uncased`** for binary sentiment classification, evaluates the model, and runs quick predictions.

**Outline**
1. Install dependencies
2. Imports & environment setup
3. Download dataset (kagglehub)
4. Load & preprocess data
5. Train/validation/test split (80/10/10, stratified)
6. Tokenizer, PyTorch dataset & collator
7. Model & metrics
8. TrainingArguments & Trainer
9. Train & evaluate
10. Quick predictions


In [ ]:
# (1) Install dependencies
# Note: We keep the current PyTorch in Colab to avoid CUDA mismatches.
%pip -q install -U kagglehub transformers scikit-learn pandas


In [ ]:
# (2) Imports & environment setup
import os, glob, inspect
import numpy as np
import pandas as pd
import kagglehub
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, set_seed
)

# Optional: disable Weights & Biases auto-logging in case the environment enables it by default
os.environ["WANDB_DISABLED"] = "true"

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


In [ ]:
# TODO: Load the IMDb dataset
# (3) Load the IMDb dataset using the 🤗 Datasets library. Create train/test splits (and optionally a
# small validation set). Name the resulting object `raw_datasets` so later cells can use it.

from datasets import load_dataset

# raw_datasets = load_dataset('imdb')
raw_datasets = None  # replace None after you implement

In [ ]:
# (4) Load & preprocess data (binary labels: negative=0, positive=1)
df = pd.read_csv(csv_path)
expected_cols = {"review", "sentiment"}
if not expected_cols.issubset(set(df.columns)):
    raise ValueError("Unexpected CSV format. Expected columns: 'review', 'sentiment'.")

# Basic cleaning: ensure strings and map labels
df["text"] = df["review"].astype(str)
df["label"] = (df["sentiment"].str.lower() == "positive").astype(int)
df = df[["text", "label"]]

df.head()


In [ ]:
# (5) Train/validation/test split (80/10/10), stratified over the whole dataset
df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["label"], random_state=42
)

# Optional: tidy indices
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Splits -> train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")


In [ ]:
# TODO: Instantiate the tokenizer
# (6) Create a BERT tokenizer (e.g., 'bert-base-uncased') with `AutoTokenizer.from_pretrained`. Name
# it `tokenizer`. Consider setting `use_fast=True`.

from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased', use_fast=True)
tokenizer = None  # replace None after you implement

In [ ]:
# TODO: Create the classification model
# (7) Load a `AutoModelForSequenceClassification` checkpoint (e.g., 'bert-base-uncased') with
# `num_labels=2`. Name the model `model`.

from transformers import AutoModelForSequenceClassification

# model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model = None  # replace after you implement

In [ ]:
# TODO: Create a data collator
# (8) Instantiate a data collator compatible with BERT for dynamic padding (e.g.,
# `DataCollatorWithPadding`). Name it `data_collator`.

from transformers import DataCollatorWithPadding

# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator = None  # replace after you implement

In [ ]:
# TODO: Train the model
# (9) Call `trainer.train()` to fine-tune the model.

# trainer.train()
raise NotImplementedError('Call trainer.train() once everything above is ready')

In [ ]:
# (10) Quick predictions
def predict(texts):
    batch = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(**batch).logits
    return [id2label[i] for i in logits.argmax(dim=-1).cpu().tolist()]

print(predict([
    "Absolutely fantastic movie. I loved every minute!",
    "It was okay, not great, not terrible.",
    "Terrible plot and bad acting."
]))
